# 🏦 JPMC Consumer Banking - Hands-on Lab
## Notebook 08b: Spark on SPCS - Iceberg via Horizon REST Catalog

---

### What You'll Learn
- Running **Apache Spark** in a Snowpark Container Services (SPCS) container
- Configuring Spark to use **Snowflake Horizon REST Catalog** for Iceberg metadata
- Querying **Managed Iceberg tables WITHOUT Snowflake compute** (no warehouse used)
- Bi-directional interoperability between Spark and Snowflake

### Architecture
```
┌──────────────────────────────────┐
│   SPCS Container (Spark 3.5)     │
│   ┌────────────────────────┐     │
│   │  PySpark + Iceberg Lib │     │   ← Container compute (CPU_X64_M)
│   └──────────┬─────────────┘     │
└──────────────┼───────────────────┘
               │ REST API calls
               ▼
┌──────────────────────────────────┐
│   Snowflake Horizon REST Catalog │   ← Metadata only (no warehouse)
│   - Table discovery              │
│   - Schema information           │
│   - Data file locations          │
└──────────────┬───────────────────┘
               │ Direct file read
               ▼
┌──────────────────────────────────┐
│   Cloud Object Storage           │   ← Iceberg data files (Parquet)
│   (S3 / Azure Blob / GCS)       │
└──────────────────────────────────┘
```

> **Runtime:** Container (SPCS) - NOT a warehouse-backed notebook
> **Compute Pool:** `HOL_SPARK_POOL` (CPU_X64_M, 1-2 nodes)
> **External Access:** `HOL_MAVEN_EAI` (created in Notebook 08a)

## Step 1: Install Dependencies (Java + PySpark)

This cell installs Java (required by Spark) and PySpark. Internet access is provided by the `HOL_MAVEN_EAI` external access integration attached in Notebook 08a.

In [ ]:
import subprocess, os

# Verify/Install Java
result = subprocess.run(["bash", "-c", "which java"], capture_output=True, text=True)
if result.returncode == 0:
    java_bin = result.stdout.strip()
    resolve = subprocess.run(["bash", "-c", f"dirname $(dirname $(readlink -f {java_bin}))"],
                            capture_output=True, text=True)
    os.environ["JAVA_HOME"] = resolve.stdout.strip()
    print(f"✅ JAVA_HOME = {os.environ['JAVA_HOME']}")
else:
    # Install Java (requires HOL_MAVEN_EAI external access)
    print("Installing Java...")
    subprocess.run(["bash", "-c", "apt-get update -qq && apt-get install -y -qq default-jdk > /dev/null 2>&1"], check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    print(f"✅ Java installed. JAVA_HOME = {os.environ['JAVA_HOME']}")

# Install PySpark
subprocess.run(["pip", "install", "pyspark==3.5.1", "--quiet"], check=True)
print("✅ PySpark 3.5.1 installed")

# Verify connectivity to Maven (needed for Iceberg JAR download)
import urllib.request
try:
    urllib.request.urlopen("https://repo1.maven.org", timeout=5)
    print("✅ Maven Central is reachable — Iceberg JARs will download at Spark startup")
except Exception as e:
    print(f"❌ Cannot reach Maven Central: {e}")
    print("   Make sure HOL_MAVEN_EAI is attached to this notebook (done in 08a)")

## Step 2: Configure Spark Session with Iceberg + Horizon REST Catalog

In [ ]:
# =============================================================================
# SPARK SESSION CONFIGURATION
# Configure Spark with Iceberg + Snowflake Horizon REST Catalog
# =============================================================================

from pyspark.sql import SparkSession
import os

# Get Snowflake connection details from container environment
# (SPCS containers have access tokens automatically injected)
SNOWFLAKE_ACCOUNT = os.environ.get("SNOWFLAKE_ACCOUNT", "<your-account>")
SNOWFLAKE_HOST = os.environ.get("SNOWFLAKE_HOST", f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com")
SNOWFLAKE_TOKEN = os.environ.get("SNOWFLAKE_TOKEN", "")  # Auto-injected in SPCS

# Horizon REST Catalog endpoint
CATALOG_URI = f"https://{SNOWFLAKE_HOST}/polaris/api/catalog"

# Build Spark session with Iceberg extensions
# spark.jars.packages downloads JARs from Maven Central (requires HOL_MAVEN_EAI)
spark = SparkSession.builder \
    .appName("JPMC_HOL_Iceberg_Interop") \
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
            "org.apache.iceberg:iceberg-rest-catalog-api:1.5.0") \
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.snowflake_catalog",
            "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.snowflake_catalog.type", "rest") \
    .config("spark.sql.catalog.snowflake_catalog.uri", CATALOG_URI) \
    .config("spark.sql.catalog.snowflake_catalog.credential", SNOWFLAKE_TOKEN) \
    .config("spark.sql.catalog.snowflake_catalog.warehouse",
            "JPMC_CONSUMER_BANKING_HOL") \
    .config("spark.sql.catalog.snowflake_catalog.scope",
            "PRINCIPAL_ROLE:ALL") \
    .config("spark.sql.defaultCatalog", "snowflake_catalog") \
    .getOrCreate()

print(f"✅ Spark session created: {spark.version}")
print(f"   Catalog URI: {CATALOG_URI}")
print(f"   Database: JPMC_CONSUMER_BANKING_HOL")

## Step 3: Discover Tables via Horizon Catalog

In [ ]:
# =============================================================================
# DISCOVER TABLES via Horizon REST Catalog
# No Snowflake warehouse is used for any of these operations!
# =============================================================================

# List available namespaces (schemas)
spark.sql("SHOW NAMESPACES IN snowflake_catalog").show()

# List tables in HOL_LAB schema
spark.sql("SHOW TABLES IN snowflake_catalog.HOL_LAB").show()

# Note: Only Iceberg tables are visible through the REST catalog
# Native Snowflake tables (CUSTOMERS, TRANSACTIONS, SUPPORT_TICKETS) 
# are NOT accessible via this path - by design!

## Step 4: Read Iceberg Tables with Spark (No Snowflake Warehouse!)

In [ ]:
# =============================================================================
# READ ICEBERG TABLES DIRECTLY WITH SPARK
# Zero Snowflake warehouse credits consumed!
# =============================================================================

# Read ACCOUNTS table (managed Iceberg)
accounts_df = spark.read.table("snowflake_catalog.HOL_LAB.ACCOUNTS")
print(f"ACCOUNTS: {accounts_df.count()} rows, {len(accounts_df.columns)} columns")
accounts_df.printSchema()
accounts_df.show(5)

In [ ]:
# Read LOANS table
loans_df = spark.read.table("snowflake_catalog.HOL_LAB.LOANS")
print(f"LOANS: {loans_df.count()} rows")

# Read CREDIT_CARDS table
credit_cards_df = spark.read.table("snowflake_catalog.HOL_LAB.CREDIT_CARDS")
print(f"CREDIT_CARDS: {credit_cards_df.count()} rows")

## Step 5: Run Spark Transformations on Iceberg Data

In [ ]:
# =============================================================================
# SPARK ANALYTICS ON ICEBERG TABLES
# Same data that Snowflake DTs process, now processed by Spark!
# =============================================================================

from pyspark.sql import functions as F

# Analysis 1: Portfolio summary across all three Iceberg tables
portfolio_summary = accounts_df.join(loans_df, "CUSTOMER_ID", "left") \
    .join(credit_cards_df, "CUSTOMER_ID", "left") \
    .groupBy("ACCOUNT_TYPE") \
    .agg(
        F.count("*").alias("total_records"),
        F.sum("BALANCE").alias("total_account_balance"),
        F.sum(loans_df["CURRENT_BALANCE"]).alias("total_loan_balance"),
        F.sum(credit_cards_df["CURRENT_BALANCE"]).alias("total_cc_balance"),
        F.avg("INTEREST_RATE").alias("avg_account_rate")
    )

print("Portfolio Summary (computed by Spark, data from Snowflake Iceberg):")
portfolio_summary.show()

In [ ]:
# Analysis 2: Loan risk distribution (same as Snowflake DT LOAN_RISK_PROFILE)
loan_risk_spark = loans_df.withColumn(
    "risk_category",
    F.when(F.col("STATUS").isin("DEFAULT"), "CRITICAL")
     .when(F.col("STATUS").isin("DELINQUENT_90"), "CRITICAL")
     .when(F.col("STATUS").isin("DELINQUENT_60", "DELINQUENT_30"), "HIGH")
     .when(F.col("DEBT_TO_INCOME_RATIO") > 0.45, "MEDIUM")
     .otherwise("LOW")
)

print("Loan Risk Distribution (Spark-computed):")
loan_risk_spark.groupBy("risk_category") \
    .agg(
        F.count("*").alias("loan_count"),
        F.sum("CURRENT_BALANCE").alias("total_exposure"),
        F.avg("INTEREST_RATE").alias("avg_rate")
    ) \
    .orderBy("total_exposure", ascending=False) \
    .show()

In [ ]:
# Analysis 3: Use SparkSQL for familiar SQL syntax on Iceberg tables
spark.sql("""
    SELECT 
        LOAN_TYPE,
        STATUS,
        COUNT(*) as loan_count,
        ROUND(AVG(PRINCIPAL_AMOUNT), 2) as avg_principal,
        ROUND(AVG(CURRENT_BALANCE), 2) as avg_balance,
        ROUND(AVG(INTEREST_RATE), 3) as avg_rate
    FROM snowflake_catalog.HOL_LAB.LOANS
    GROUP BY LOAN_TYPE, STATUS
    HAVING COUNT(*) > 100
    ORDER BY LOAN_TYPE, loan_count DESC
""").show(20)

## Step 6: Write Results Back to Iceberg (Bi-directional)

In [ ]:
# =============================================================================
# WRITE BACK TO ICEBERG (Spark → Snowflake readable)
# =============================================================================

# Create a Spark-computed analytics table and write it as Iceberg
customer_portfolio = accounts_df \
    .groupBy("CUSTOMER_ID", "ACCOUNT_TYPE") \
    .agg(
        F.sum("BALANCE").alias("TOTAL_BALANCE"),
        F.count("*").alias("ACCOUNT_COUNT"),
        F.max("LAST_ACTIVITY_DATE").alias("LAST_ACTIVITY")
    ) \
    .withColumn("COMPUTED_BY", F.lit("SPARK_ON_SPCS")) \
    .withColumn("COMPUTED_AT", F.current_timestamp())

# Write as a new Iceberg table via the catalog
customer_portfolio.writeTo("snowflake_catalog.HOL_LAB.SPARK_CUSTOMER_PORTFOLIO") \
    .using("iceberg") \
    .createOrReplace()

print("✅ Spark wrote SPARK_CUSTOMER_PORTFOLIO table")
print(f"   Rows written: {customer_portfolio.count()}")
print("   This table is now queryable from Snowflake!")

## Step 7: Verify from Snowflake Side

In [ ]:
-- =============================================================================
-- VERIFY: Query Spark-written table from Snowflake
-- =============================================================================

-- The table written by Spark is now queryable using Snowflake SQL!
SELECT * 
FROM SPARK_CUSTOMER_PORTFOLIO
WHERE COMPUTED_BY = 'SPARK_ON_SPCS'
LIMIT 10;

-- Count rows
SELECT COUNT(*) AS spark_written_rows FROM SPARK_CUSTOMER_PORTFOLIO;

## Step 8: Compare - Same Analysis, Different Engines

| Metric | Snowflake DT (NB 03) | Spark on SPCS (NB 08b) |
|--------|----------------------|----------------------|
| **Data Format** | Read from Iceberg | Read from Iceberg |
| **Compute** | Snowflake Warehouse | SPCS Container (Spark) |
| **Catalog** | Snowflake Internal | Horizon REST Catalog |
| **Result** | Same analytical output | Same analytical output |
| **Cost Model** | Credits/second (warehouse) | Credits/hour (container) |
| **Best For** | SQL-centric teams | Existing Spark pipelines |
| **Latency** | Sub-second for cached | Startup time + processing |

### Key Takeaway
With Managed Iceberg Tables + Horizon Catalog:
- **One copy of data** serves multiple engines
- **No data movement** between Snowflake and Spark
- **Choose the right engine** for each workload without data silos
- **Governance stays unified** through Snowflake's access control

In [ ]:
# Cleanup: Stop Spark session
spark.stop()
print("✅ Spark session stopped")